# 4장 1강: LLM API 활용 — 실습문제


## 실습 목표

- LLM API의 역할과 입력 준비·요청·처리·응답·확인 흐름을 설명할 수 있다.
- `model`, `instructions`, `input`의 역할을 구분하여 요청 정보를 구성할 수 있다.
- 응답의 `id`, `status`, `output`, `usage`를 읽을 수 있다.
- 요청 처리 성공, 결과 존재, 출력 형식, 분석 정확성을 구분하여 검증할 수 있다.
- 원문·기존 출력·새 응답을 구분하고 실패 상태와 확인된 사용량을 관리할 수 있다.
- 인증 정보, 오류 원인, 비용, 입력 데이터 활용 시 확인할 사항을 설명할 수 있다.

## 실습 환경 / 데이터

- 기본 실습: Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- 세 CSV를 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 이전 노트북을 실행할 필요는 없습니다. 이 파일의 준비 코드부터 순서대로 실행합니다.

| 파일 | 활용 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건 분류의 입력·지시·기존 출력을 확인하고 API 요청 구성 |
| 엘지유플러스_요약.csv | 요약 응답이 원문과 출력 조건을 충족하는지 검토 |
| 엘지유플러스_질의응답.csv | 특정 사실을 묻는 요청과 답변을 원문에 대조 |

세 파일의 공통 컬럼은 `source_id`, `consulting_category`, `task_category`, `instruction`, `input`, `output`입니다. `source_id`는 상담 원문 ID이며, 같은 상담에 여러 작업·질문이 연결되므로 파일과 세부 작업도 확인합니다. `instruction`은 기존 지시, `input`은 원문, `output`은 파일에 이미 저장된 출력입니다.

### 기본 실습과 실제 호출 구분

교안의 요청·응답 구조와 결과 확인을 먼저 익힐 수 있도록 **필수·과제는 가상 응답으로 실행**합니다. 노트북 끝에는 같은 내용을 실제 API 단일 요청에 연결하는 **선택 실행용 제공 코드**가 있습니다.

- 원문·지시·기존 출력은 실제 CSV 값입니다. 기존 출력도 원문과 비교할 참고 자료입니다.
- 기본 실습의 응답 ID·응답 텍스트·HTTP 코드·사용 토큰 수는 작성자가 정한 가상 값입니다. 실제 모델 실행·토큰 측정·결제 결과가 아닙니다.
- `model`의 `사용 가능한 모델 ID`는 구조 설명용 표시입니다. 실제 호출에서는 본인 계정에서 사용할 수 있는 모델 ID로 바꿉니다.
- 실제 호출은 기본적으로 꺼져 있습니다. 선택 실행에는 `openai` 패키지와 사용 가능한 API 키·모델이 필요하며, 상담 원문이 외부 API로 전송되고 비용이 발생할 수 있습니다.
- API 키는 코드·출력·공유 파일에 적지 않습니다. 환경변수 또는 선택 코드의 비공개 입력을 사용합니다.
- 상세 일괄 처리 파이프라인, 자동 재시도 로직, 구조화된 출력 강제, 요금표 기반 비용 계산은 요구하지 않습니다.
- `<NAME>`, `<DATE>`, `<CHARGE>` 같은 가림 표시는 그대로 유지합니다. 실습의 원문 해석을 현재 통신사 정책으로 일반화하지 않습니다.

**난이도와 구성:** 기초~중급, 필수 2문제와 같은 수준의 과제 1문제입니다. 제공 함수의 구현을 암기하기보다 요청과 응답을 읽고 판단하는 데 집중합니다.


## 시작 전에 읽는 개념 정리

### 1. 이번 강의에서 배우는 연결 관계

앞 강의에서는 원문과 지시를 어떻게 작성하고 결과를 어떻게 검토할지 살펴보았습니다. 이번에는 **파이썬 프로그램이 지시와 원문을 API에 전달하고, 돌아온 응답을 표에 연결하는 과정**을 배웁니다. API는 프로그램끼리 정해진 방식으로 요청과 응답을 주고받는 연결 방법입니다. LLM API는 그 연결을 통해 언어 모델의 기능을 프로그램에서 사용할 수 있게 합니다.

대화창에서는 사람이 내용을 입력하고 답을 읽습니다. API를 이용하면 프로그램이 데이터를 읽어 요청을 구성하고, 응답의 상태·텍스트·사용량을 받아 후속 처리에 연결할 수 있습니다. **API를 사용한다는 사실이 생성 내용의 정확성까지 보장하지는 않습니다.** 원문과 작업 조건을 바탕으로 검토하는 절차가 여전히 필요합니다.

| 흐름 | 실제 호출에서의 의미 | 이번 기본 실습에서 하는 일 |
|---|---|---|
| 입력 준비 | 원문과 작업 지시를 정리 | CSV에서 한 작업 행을 선택 |
| 요청 | 모델·지시·원문을 API에 전달 | 요청 딕셔너리를 구성하되 전송하지 않음 |
| LLM 처리 | 서비스가 요청을 처리하고 결과 생성 | 실제 모델 처리는 하지 않음 |
| 응답 | 상태·출력·사용량 등의 정보 반환 | 제공 함수로 가상 응답 구조를 만듦 |
| 결과 확인 | 처리 상태와 생성 내용을 점검 | 가상 응답을 읽고 원문·조건과 대조 |

따라서 기본 모드에서 정답 코드가 실행되었다는 말은 **데이터 조회와 가상 응답 점검 코드가 실행되었다**는 뜻입니다. 모델을 호출하여 정답을 얻었다는 뜻으로 해석하지 않습니다.

### 2. 요청과 응답에서 처음 만나는 용어

| 용어 | 쉬운 설명 | 이번 코드에서 읽는 위치 |
|---|---|---|
| 요청 | 프로그램이 서비스에 보내려는 작업 정보 | `request_body`, `qa_request`, `summary_request` |
| 모델 ID | 어떤 모델로 처리할지 지정하는 이름 | 요청의 `model` |
| 지시 | 해야 할 작업과 지켜야 할 조건 | 요청의 `instructions` |
| 입력 본문 | 분석 대상이 되는 상담 원문 | 요청의 `input` |
| API Key | 서비스 이용을 인증하는 비밀 정보 | 실제 호출용 별도 설정 |
| 응답 | 요청 처리 후 돌아오는 정보 묶음 | 기본 모드의 `demo_response` |
| 응답 ID | 특정 응답을 구별하는 식별 정보 | 응답의 `id` |
| 처리 상태 | 응답 처리의 진행·완료 상태를 나타내는 정보 | 응답의 `status` |
| 생성 출력 | 모델이 생성한 결과에 해당하는 부분 | 응답의 `output` 안의 텍스트 |
| 사용량 | 처리한 입력·출력 등의 토큰 수 | 응답의 `usage` |
| HTTP 상태 코드 | 요청 처리 결과를 전달하는 통신상의 코드 | 과제 표의 `HTTP코드` |
| SDK | API를 프로그램에서 쓰도록 돕는 도구 묶음 | 선택 실행의 `OpenAI` |

`HTTP코드`와 `status`는 서로 다른 정보입니다. 과제에서는 HTTP 200과 응답 상태 completed를 함께 제시하지만, 같은 필드를 두 번 적은 것이 아닙니다. 기본 실습의 가상 응답 함수에는 HTTP 코드가 들어 있지 않습니다. 필수 2의 HTTP 200은 문제에서 정한 상황이고, 과제의 HTTP 코드는 따로 만든 처리 기록입니다.

### 3. 파이썬에서 요청·응답을 담는 방법

딕셔너리는 **이름과 값의 짝**을 저장합니다. `request_body["model"]`은 딕셔너리에서 `model`이라는 이름에 연결된 값을 꺼냅니다. 목록은 여러 값을 순서대로 담습니다. 응답의 `output`은 목록이고, 그 안의 각 항목과 `content` 안의 내용은 다시 딕셔너리로 표현됩니다. 하나의 응답이 여러 층으로 구성될 수 있다는 점을 먼저 이해하면 제공 함수를 읽기 쉽습니다.

JSON은 이런 이름·값·목록 구조를 표현하는 데이터 형식입니다. 파이썬 딕셔너리와 비슷하게 보이지만 같은 객체는 아닙니다. `json.dumps()`는 딕셔너리를 JSON 모양의 문자열로 바꾸어 보여 주는 데 사용합니다. 이 함수만 실행한다고 API로 요청이 전송되지는 않습니다.

### 4. 네 가지 확인 질문을 구분하기

| 확인 단계 | 스스로 물어볼 질문 | 이 단계만으로는 알 수 없는 것 |
|---|---|---|
| 요청·처리 상태 | 요청이 처리되었고 필요한 응답 상태인가? | 텍스트가 실제로 있는지, 내용이 맞는지 |
| 결과 존재 | 비어 있지 않은 결과가 있는가? | 요청한 형식인지, 원문과 일치하는지 |
| 출력 형식 | 범주명·문장 수·길이 등 조건을 지켰는가? | 선택한 범주나 사실이 정확한지 |
| 분석 정확성 | 원문과 분류 기준에 맞는 내용인가? | 다른 요청도 모두 정확하다고 일반화할 수 있는지 |

`True`라는 출력도 **어떤 조건을 검사한 결과인지** 함께 읽습니다. 빈 문자열인지 검사한 `True`는 ‘내용이 맞다’는 판정이 아닙니다. 형식 검사를 통과해도 내용 검토에서 수정이 필요할 수 있습니다.

### 5. 이름이 비슷해도 서로 다른 값

| 구분 | 의미 |
|---|---|
| CSV의 `output` | 파일에 이미 저장되어 있는 분류·요약·답변 |
| 가상 응답의 `output` | 메시지와 내용 항목을 담은 목록 |
| 내용 항목의 `type: output_text` | 해당 항목이 출력 텍스트임을 나타내는 표시 |
| SDK의 `response.output_text` | 생성 텍스트를 읽기 위한 편의 속성 |
| 결과 표의 `새_가상출력` / `새_API출력` | 새로 받은 것으로 다루는 응답을 기존 출력과 구분해 기록한 열 |

기존 출력이 새 응답과 같더라도 두 값의 출처는 다릅니다. 기존 답을 읽거나 복사한 일을 모델이 답한 것으로 기록하지 않습니다. 또한 본문 글자 수와 토큰 수는 다른 값이고, 토큰 수와 원화 비용도 같은 값이 아닙니다.


## 실습 준비

아래 **제공 코드**를 실행하세요.

1. 세 파일의 크기, 컬럼·자료형, 결측치·빈 문자열을 확인합니다.
2. 고유 상담 수, 작업 유형 분포와 본문 길이를 확인합니다.
3. 상담 40003·40005의 원문이 세 파일에서 동일한지 확인합니다.
4. `get_record()`로 정확한 작업 행을 선택하고 `show_record()`로 내용을 확인합니다.

`record_id`는 파일 이름과 0부터 시작하는 행 위치를 결합한 **실습용 행 식별자**입니다. 원본의 영구 ID나 API 응답 ID가 아닙니다. 본문 글자 수도 API 토큰 사용량이 아닙니다.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])



[분류] 원본 16,080행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
상담 요건    3216
상담 주제    3216
상담 사유    3216
상담 내용    3216
상담 결과    3216
행별 본문 글자 수(토큰 수가 아님):
min      670.00
mean    1583.42
max     2836.00

[요약] 원본 11,690행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
기본 요약          3217
길이 제한 요약       2928
주제 중점 요약       2862
핵심 정보 추출 요약    2683
행별 본문 글자 수(토큰 수가 아님):
min      215.00
mean    1576.97
max     2836.00

[질의응답] 원본 7,497행 × 6열
                       자료형  결측치
source_id        

### 준비 코드 읽기 — 파일과 표의 구조부터 확인하기

`pandas`의 `DataFrame`은 행과 열로 이루어진 표입니다. 이번 데이터의 한 행은 ‘상담 한 개’ 자체보다 **특정 상담에 대한 작업 한 개**로 이해하면 좋습니다. 같은 상담에 분류·요약·질문을 여러 개 만들 수 있기 때문입니다.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | 데이터가 다른 폴더라면 해당 폴더로 수정 |
| `DATA_DIR / "엘지유플러스_분류.csv"` | 폴더와 파일 이름을 연결 | `/`는 여기서 경로 연결 역할 |
| `pd.read_csv(...)` | CSV를 표로 읽음 | 파일 조회이며 LLM 호출이 아님 |
| `tables = {"분류": classification, ...}` | 이름과 표를 짝지어 보관 | 세 파일에 같은 점검 절차를 적용하기 위함 |
| `tables.items()` | 이름과 표를 한 쌍씩 꺼냄 | 첫 반복에서 분류 표, 다음 반복에서 다른 표 확인 |
| `table.shape` | 표의 행 수·열 수 확인 | 행 수가 곧 고유 상담 수인 것은 아님 |
| `table.dtypes.astype(str)` | 열별 자료형 이름을 문자열로 표시 | 원본 상담 본문을 변환하는 작업이 아님 |
| `table.isna().sum()` | 열별 결측치 수 계산 | 비어 있는 값 확인 |
| `table["source_id"].nunique()` | 중복을 제외한 상담 ID 개수 계산 | 동일 ID의 작업 여러 행을 한 상담으로 셈 |
| `.str.strip().eq("").sum()` | 앞뒤 공백을 제외하면 빈 문자열인 값의 수 계산 | 결측치 검사와 별도로 내용 없는 문자열 확인 |
| `.value_counts()` | 작업 유형마다 몇 행인지 계산 | 상담 수보다 작업의 구성 확인에 초점 |
| `.str.len()` | 각 행의 본문 글자 수 계산 | 토큰 수를 계산하는 기능은 아님 |
| `.agg(["min", "mean", "max"])` | 길이의 최솟값·평균·최댓값 계산 | 파일에 있는 각 행을 기준으로 집계 |
| `.round(2)` | 계산값을 소수점 둘째 자리까지 반올림 | 본문을 자르거나 요약하는 기능이 아님 |

`print()`는 값을 화면에 보여 줍니다. `f"...{값}..."`는 문자열 중간에 값을 넣는 표기입니다. 예를 들어 `{table.shape[0]:,}`의 쉼표는 큰 행 수를 천 단위로 나누어 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 출력하면서 왼쪽 인덱스를 숨기는 옵션이며, ID 열이나 실제 행을 삭제하지 않습니다.

### 같은 상담에 여러 작업이 있을 때의 ID

- `source_id`: 어느 상담에서 나온 자료인지 구분합니다.
- `task_category`: 그 상담에 대해 어떤 종류의 작업을 수행하는지 나타냅니다.
- `instruction`: 실제로 요구한 작업이나 질문 문장입니다.
- `record_id`: 현재 파일의 작업 행을 구별하기 위해 실습 코드가 추가한 이름입니다.

`[f"{name}-{i}" for i in range(len(table))]`는 현재 표의 각 행에 `분류-0`, `분류-1`처럼 이름을 붙입니다. `range(len(table))`는 0부터 행 수 직전까지의 번호를 만듭니다. 원본 표의 정렬 순서가 바뀌면 번호도 달라질 수 있으므로 원본 상담 ID와 같은 의미로 쓰지 않습니다.

준비 코드가 `record_id`를 추가한 뒤 표는 7열이 됩니다. 출력되는 ‘원본 6열’은 그 열을 추가하기 전에 확인한 값입니다. CSV 파일에 새 열을 저장하는 `to_csv()`는 실행하지 않습니다.


### 응답 구조 확인용 제공 함수

아래 함수는 **기본 실습용 보조 도구**입니다. 외부 API를 호출하지 않습니다.

- `make_demo_response()`는 교안의 축약 구조를 본뜬 가상 응답을 만듭니다. 실제 응답의 모든 필드를 재현하지 않습니다.
- `read_demo_text()`는 `output`의 각 항목에서 메시지와 `output_text` 내용을 확인해 텍스트를 읽습니다. 첫 번째 항목·첫 번째 내용에 항상 정답이 있다고 가정하지 않습니다.
- 실제 Python SDK 객체에서는 교안의 `response.output_text`로 생성 텍스트를 확인할 수 있습니다. 기본 실습의 딕셔너리 자체에는 그 편의 속성이 없습니다.


In [2]:
def make_demo_response(response_id, text, input_tokens, output_tokens):
    # 학습용 축약 응답입니다. text=None은 출력이 없는 가정을 뜻합니다.
    items = []
    if text is not None:
        items = [{"type": "message", "role": "assistant",
                  "content": [{"type": "output_text", "text": text}]}]
    return {"id": response_id, "status": "completed", "output": items,
            "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens,
                      "total_tokens": input_tokens + output_tokens}}

def read_demo_text(response_data):
    texts = []
    for item in response_data.get("output", []):
        if item.get("type") == "message":
            for content in item.get("content", []):
                if content.get("type") == "output_text":
                    texts.append(content.get("text", ""))
    return "".join(texts)


### 제공 함수 상세 설명 — 응답을 만들고 텍스트를 읽는 순서

이 함수들은 작성 연습의 평가 대상이 아니라 **응답 구조를 살펴보기 위한 제공 도구**입니다. 함수 이름, 전달하는 값, 반환값부터 연결해서 읽으면 됩니다. `def`는 함수를 정의하고, `return`은 처리 결과를 호출한 곳에 돌려줍니다. 함수 정의 셀을 실행하는 것만으로 실제 응답이 만들어지지는 않습니다. 아래 문제에서 함수를 호출할 때 값이 만들어집니다.

#### `make_demo_response()`는 무엇을 만드는가?

| 코드 | 역할 |
|---|---|
| `response_id, text, input_tokens, output_tokens` | 호출할 때 전달하는 응답 이름·텍스트·가상 입력량·가상 출력량 |
| `items = []` | 출력 항목을 담을 빈 목록 생성 |
| `if text is not None:` | 텍스트 값이 제공되었는지 확인 |
| `type: message`, `role: assistant` | 출력 항목이 assistant 메시지라는 구조를 표시 |
| `content` 안의 `type: output_text`, `text` | 텍스트 내용의 종류와 실제 문자열을 넣음 |
| `status: completed` | 연습용 완료 상태를 고정해 넣음 |
| `input_tokens + output_tokens` | 전달받은 두 숫자를 더해 가상 전체 사용량을 만듦 |

이 함수에는 입력 문장을 모델에 보내거나 토큰으로 세는 과정이 없습니다. `input_tokens`에 숫자를 전달하면 그 숫자를 그대로 저장합니다. `status`도 외부 서버에 물어본 값이 아니라 함수에 적힌 값입니다. 교안의 핵심 구조만 보여 주는 축약 예시이며 실제 API 응답의 모든 경우를 재현하지 않습니다.

#### `read_demo_text()`는 어느 부분을 읽는가?

| 순서 | 코드·조건 | 동작 |
|---:|---|---|
| 1 | `texts = []` | 찾은 텍스트들을 모을 목록 준비 |
| 2 | `response_data.get("output", [])` | output 목록을 읽고, 이름이 없으면 빈 목록 사용 |
| 3 | `item.get("type") == "message"` | 메시지에 해당하는 항목만 확인 |
| 4 | `item.get("content", [])` | 그 메시지의 내용 목록을 하나씩 읽음 |
| 5 | `content.get("type") == "output_text"` | 텍스트 출력에 해당하는 내용만 선택 |
| 6 | `texts.append(content.get("text", ""))` | 찾은 문자열을 목록 뒤에 추가 |
| 7 | `"".join(texts)` | 모은 문자열을 사이 구분자 없이 이어 반환 |

`.get(이름, 기본값)`은 딕셔너리에 그 이름이 없을 때 기본값을 쓰는 방식입니다. 이름은 있는데 값이 `None`인 경우까지 자동으로 기본값으로 바꾸는 기능은 아닙니다. 여기서는 위 함수가 만든 정해진 구조를 읽는 데 사용합니다. 실제 모든 응답 오류를 처리하는 범용 검사 함수로 해석하지 않습니다.

바깥 `for`는 출력 항목을, 안쪽 `for`는 해당 메시지의 내용을 순서대로 봅니다. 첫 번째 위치가 늘 텍스트라는 가정을 피하기 위한 구조입니다. 다만 텍스트를 잘 찾아냈다는 사실과 그 텍스트가 정답이라는 사실은 별개입니다.

#### `None`과 빈 문자열은 어떻게 다른가?

| 전달한 `text` | 가상 응답의 구조 | 텍스트를 읽은 결과 |
|---|---|---|
| 일반 문자열 | 메시지와 텍스트 내용 항목을 만듦 | 해당 문자열 |
| `None` | `output`을 빈 목록으로 둠 | 빈 문자열 `""` |
| 빈 문자열 `""` | 메시지 구조는 있지만 텍스트가 비어 있음 | 빈 문자열 `""` |

`None`은 값이 없음을 나타내는 파이썬 값이고 `""`는 길이가 0인 문자열입니다. 서로 다른 입력이지만 이 텍스트 읽기 함수에서는 둘 다 비어 있는 결과가 될 수 있습니다. 그래서 출력 존재 검사만으로 원래 어떤 구조였는지까지 알 수는 없습니다. 필요할 때는 응답 구조와 상태도 함께 확인합니다.


---
## 필수 1. API 요청 구성과 응답 항목 읽기

### 문제 1-1. 상담 분류 요청과 가상 응답 연결하기

#### 문제 설명

상담 ID **40005**의 `상담 요건`을 `단일 요건 민원`, `다수 요건 민원` 중 하나로 분류합니다. 원문에는 통화 중 대기 서비스 문제와 추가 결합 할인 상담이 함께 있습니다.

요청에는 다음 항목을 사용합니다.

| 요청 항목 | 작성할 내용 |
|---|---|
| model | 구조 확인용 문자열 `사용 가능한 모델 ID` |
| instructions | 분류 작업·두 범주 정의·하나만 선택·범주명만 출력 조건 |
| input | 선택한 상담의 원문 전체 |

이번 기준에서 단일은 중심 요청·주제가 하나인 경우, 다수는 서로 구분되는 요청·주제가 둘 이상인 경우입니다. 한 요청의 절차를 여러 번 질문한 것만으로 다수라고 보지는 않습니다.

다음 **가상 응답 값**을 사용하세요.

| 항목 | 값 |
|---|---|
| id | resp_class_demo |
| status | completed |
| 생성 텍스트 | 다수 요건 민원 |
| input_tokens | 1200 |
| output_tokens | 8 |
| total_tokens | 1208 |

#### 요구사항

1. `class_row = get_record(classification, 40005, "상담 요건")`으로 작업 행을 가져오고 원문을 확인하세요.
2. `model`, `instructions`, `input`의 세 항목을 가진 딕셔너리 `request_body`를 작성하세요. 인증 키와 CSV의 기존 정답 `output`을 본문에 넣지 않습니다.
3. `make_demo_response("resp_class_demo", "다수 요건 민원", 1200, 8)`로 `demo_response`를 만드세요. 응답 전체와 `id`, `status`, 생성 텍스트, `usage`를 각각 확인합니다.
4. 처리 완료 여부, 비어 있지 않은 텍스트 여부, 허용 범주 여부를 확인하세요. 실제 원문에 맞는지는 별도로 읽고 판단합니다.
5. `source_id`, `record_id`, `원문`, `기존_output`, `response_id`, `status`, `새_가상출력`, `input_tokens`, `output_tokens`, `total_tokens`의 한 행을 가진 `result_table`을 만드세요. 원문과 기존 출력은 덮어쓰지 않습니다.

#### 해석 질문

**Q1.** LLM API의 역할과 입력 준비부터 결과 확인까지의 흐름을 설명하세요. 직접 대화하는 방식과 무엇이 다른가요?

**Q2.** `model`, `instructions`, `input`, API Key의 역할을 구분하세요. 같은 기준으로 다른 상담을 분류하려면 무엇을 바꾸나요?

**Q3.** 응답 ID, 처리 상태, 생성 출력, 토큰 사용량은 각각 무엇을 뜻하나요? `source_id`와 응답 `id`, 원본 JSON의 `output`과 SDK의 `output_text`는 어떻게 다른가요?

**Q4.** 응답이 completed이고 허용 범주라면 정확성도 보장되나요? 이번 출력의 실제 근거와 토큰 수 1208을 해석할 때의 주의점을 설명하세요.

#### 제출 결과

- 요청·응답 확인 코드
- 원문과 기존 출력이 보존된 `result_table`
- Q1~Q4 답변


### 풀이 안내 — 요청 구성과 응답 읽기를 나누어 생각하기

먼저 준비 코드와 제공 함수 셀을 실행합니다. 그다음 상담 행을 선택하고, 선택한 행의 `input`을 요청에 연결합니다. `instruction`은 CSV 열 이름이고 `instructions`는 이번 API 요청 항목 이름이므로 철자를 구분하세요.

| 작업 | 확인할 점 |
|---|---|
| 요청 작성 | 어떤 모델인지, 어떤 작업인지, 어떤 원문인지 세 항목을 나누었는가? |
| 응답 읽기 | 응답 이름·상태·텍스트·사용량을 구분해 꺼냈는가? |
| 조건 확인 | 완료 여부·텍스트 존재·범주 조건을 따로 확인했는가? |
| 결과 연결 | 기존 출력과 새 응답을 다른 열에 남겼는가? |
| 해석 작성 | 원문의 어떤 대목이 분류 판단을 뒷받침하는가? |

`bool(문자열.strip())`는 앞뒤 공백을 제외하고 내용이 있는지 살펴보는 구문입니다. `값 in 목록`은 값이 목록의 항목 중 하나와 일치하는지 확인합니다. 두 구문 모두 원문의 의미를 이해하는 기능은 아닙니다. 해석 답변에는 **확인한 결과 → 해당 결과가 뜻하는 범위 → 원문 근거**를 연결해 적으세요.


In [3]:
# 필수 1 코드를 작성하세요.
import json

# 1. 상담 40005의 상담 요건 행 선택과 원문 확인
class_row = get_record(classification, 40005, "상담 요건")
show_record(class_row, include_input=True)

# 2. 요청 정보 구성 (전송하지 않음, API 키와 기존 정답 output은 넣지 않음)
request_body = {
    "model": "사용 가능한 모델 ID",
    "instructions": (
        "다음 상담의 상담 요건을 분류하세요. "
        "상담의 중심 요청이나 주제가 하나면 '단일 요건 민원', "
        "서로 구분되는 요청이나 주제가 두 가지 이상이면 '다수 요건 민원'입니다. "
        "한 요청의 절차, 준비물, 처리 시점을 여러 번 질문한 것은 하나의 요건으로 봅니다. "
        "두 범주 중 하나만 골라 범주 이름만 출력하고 설명은 붙이지 마세요."
    ),
    "input": class_row["input"],
}
# 원문은 길어서 앞부분만 보여 주고 전체 포함 여부는 따로 확인
preview = dict(request_body, input=request_body["input"][:60] + " ...(이하 생략)")
print("\n[request_body 미리보기]")
print(json.dumps(preview, ensure_ascii=False, indent=2))
print("요청 항목:", list(request_body.keys()))
print("input에 원문 전체가 들어감:", request_body["input"] == class_row["input"])
print("요청에 키 관련 항목이 없음:", not any("key" in k.lower() for k in request_body))

# 3. 가상 응답 만들고 항목별로 읽기 (실제 API 호출 아님)
demo_response = make_demo_response("resp_class_demo", "다수 요건 민원", 1200, 8)
print("\n[demo_response 전체]")
print(json.dumps(demo_response, ensure_ascii=False, indent=2))

response_id = demo_response["id"]
response_status = demo_response["status"]
new_text = read_demo_text(demo_response)
usage = demo_response["usage"]
print("\n응답 ID:", response_id)
print("처리 상태:", response_status)
print("생성 텍스트:", new_text)
print("사용량:", usage)

# 4. 단계별 확인 (내용이 맞는지는 원문을 읽고 따로 판단)
allowed_categories = ["단일 요건 민원", "다수 요건 민원"]
print("\n처리 완료(status == completed):", response_status == "completed")
print("텍스트 존재(공백 제외):", bool(new_text.strip()))
print("허용 범주 여부:", new_text in allowed_categories)

# 사람이 원문을 읽고 확인한 근거 (분류 판단용)
evidence_phrases = ["통화 중 대기 서비스 삭제를 한 번 했고요",
                    "결합 할인을 더 추가적으로 챙겨가실 수가 있는 할인 혜택"]
for p in evidence_phrases:
    print(f"원문 근거 '{p}' 존재:", p in class_row["input"])

# 5. 원문, 기존 출력, 새 가상 출력을 다른 열에 보존
result_table = pd.DataFrame([{
    "source_id": class_row["source_id"],
    "record_id": class_row["record_id"],
    "원문": class_row["input"],
    "기존_output": class_row["output"],
    "response_id": response_id,
    "status": response_status,
    "새_가상출력": new_text,
    "input_tokens": usage["input_tokens"],
    "output_tokens": usage["output_tokens"],
    "total_tokens": usage["total_tokens"],
}])
print("\n[result_table] (원문 열은 길이만 표시)")
print(result_table.assign(원문=result_table["원문"].str.len().astype(str) + "자").to_string(index=False))



record_id: 분류-23  / source_id: 40005
작업 유형: 상담 요건
지시 또는 질문: 이 상담의 내용이 주제가 하나인 "단일 요건 민원"인지, 여러 개의 주제가 있는 "다수 요건 민원"인지 알려 줘.
상담 원문:
상담사: 일상에 즐거운 변화 LGU+ <NAME>입니다.
고객: 네, 저 가입 그 뭐지 통화하고 있는 거 상대방이 알 수 있게 하는 거 있지 않습니까?
상담사: 어, 통화 중에 전화하면 대기가 걸리는 서비스 말씀하시는 거예요?
고객: 네.
상담사: 아, 네. 통화 중 대기라고 하는 서비스이고요.
고객: 네.
상담사: 그럼 신청을 하시는 우리 고객님 명의자분 성함이 <NAME> 고객님 본인 맞으실까요?
고객: <NAME>이요.
상담사: 아, 네, 본인 되십니까?
고객: 네.
상담사: 네, 반갑습니다. 잠시만 기다려 주세요.
고객: 네.
상담사: 이게 통화 중 대기 서비스가 원래 상대방한테 전화가 통화 중인 걸로 들어오면 통화 중입니다, 라고 해서 상대방한테 나오고 연결이 바로 되지가 않는데 통화 중 대기 부가 서비스를 신청을 하실 경우에는 상대방이 고객님께 전화를 걸면은 고객님한테는 뚜뚜 거리는 소리와 함께 걸려온 전화를 받을 수가 있고 상대방은 통화 중이라고 하는 멘트가 나오는 게 아니라 일반 신호음이 걸리는 거고요.
고객: 네.
상담사: 네, 근데 고객님은 지금 현재 봤을 때 통화 중 대기 부가 서비스 지금 현재 신청이 되어 있는 걸로 확인되시던데요.
고객: 아, 그래요? 근데 아까 전에 통화가 왔는데 녹취 들어가더라고요, 문자로.
상담사: 아, 문자로 그냥 매너콜 문자가 들어왔던가요, 고객님?
고객: 네.
상담사: 아, 그러면 고객님 제가 요거 통화 중 대기 서비스 삭제를 한 번 했고요.
고객: 네.
상담사: 다시 한번 신청해봐 드릴게요.
고객: 네.
상담사: 네, 고객님, 그럼 제가 테스트차 고객님한테 전화 한 번 드리겠습니다. 고객님 전화 받지 마시고요.
고객: 네.
상담사: 끊지 마시고 잠시만요.
고객: 네.
상담사: 

### 필수 1 답변 작성란

**Q1.**

- 답변: LLM API는 파이썬 같은 프로그램이 정해진 방식으로 언어 모델에 요청을 보내고 응답을 받을 수 있게 해 주는 연결 방법이다. 흐름은 입력 준비(CSV에서 상담 40005의 상담 요건 행 선택) → 요청(모델, 지시, 원문을 묶어서 전달) → LLM 처리(서비스가 결과 생성) → 응답(id, status, output, usage 반환) → 결과 확인(상태, 텍스트, 형식, 원문 대조) 순서다. 직접 대화창에서 하면 사람이 원문을 붙여 넣고 답을 눈으로 읽어야 하지만, API를 쓰면 프로그램이 데이터를 읽어서 요청을 만들고 응답의 상태나 사용량까지 표에 자동으로 연결할 수 있어서 여러 상담을 같은 방식으로 처리하기 좋다. 다만 이번 기본 실습은 요청 딕셔너리만 만들고 실제로 보내지 않았고, 응답도 `make_demo_response()`로 만든 가상 응답이다.

**Q2.**

- 답변: `model`은 어떤 모델로 처리할지 정하는 이름이고, `instructions`는 분류하라는 작업과 두 범주 정의, 하나만 고르고 범주명만 출력하라는 조건 같은 지시다. `input`은 분석 대상인 상담 원문 전체(40005는 2,297자)다. API Key는 이 서비스를 쓸 수 있는 사람인지 인증하는 비밀 정보라서 요청 본문에 넣지 않고 환경변수 같은 곳에 따로 둔다(코드로 확인했을 때 요청 항목은 model, instructions, input 세 개뿐이었다). 같은 기준으로 다른 상담을 분류하려면 `model`과 `instructions`는 그대로 두고 `input`만 그 상담의 원문으로 바꾸면 된다. 기준이 같아야 결과를 서로 비교할 수 있기 때문이다.

**Q3.**

- 답변: 응답 ID(`resp_class_demo`)는 이 응답 하나를 구별하는 이름이고, 처리 상태(`completed`)는 요청 처리가 완료되었다는 표시다. 생성 출력은 `output` 안의 `output_text`에 들어 있는 "다수 요건 민원"이고, 토큰 사용량은 입력 1200, 출력 8, 전체 1208이다(모두 가상 값). `source_id`는 원본 CSV에서 어느 상담인지(40005) 구분하는 값이고, 응답 `id`는 API 서비스가 응답마다 붙이는 이름이라서 둘은 역할이 다르다. 같은 상담을 여러 번 요청하면 source_id는 같아도 응답 id는 매번 다르다. 원본 JSON의 `output`은 메시지와 내용 항목이 여러 층으로 들어 있는 목록이고, SDK의 `output_text`는 그 안에서 텍스트만 꺼내서 바로 읽을 수 있게 해 주는 편의 속성이다. 이번 가상 응답은 딕셔너리라서 `output_text` 속성이 없어서 `read_demo_text()`로 목록을 돌면서 읽었다.

**Q4.**

- 답변: 아니다. completed는 처리가 끝났다는 뜻이고, 허용 범주 True는 두 범주 이름 중 하나를 썼다는 뜻일 뿐이라서 분류가 맞는지는 원문을 읽어야 알 수 있다. 이번 출력 "다수 요건 민원"은 원문에 맞다고 판단했다. 앞부분에서 상담사가 "통화 중 대기 서비스 삭제를 한 번 했고요", "다시 한번 신청해봐 드릴게요"라고 하며 서비스 문제를 처리했고, 뒷부분에서는 "결합 할인을 더 추가적으로 챙겨가실 수가 있는 할인 혜택"을 안내하는 별개의 주제가 나오기 때문이다. 토큰 수 1208은 문제에서 정한 가상 값이라서 실제로 측정한 사용량이 아니다. 또 원문 2,297자와 토큰 수는 단위가 달라서 글자 수로 토큰 수를 바로 계산할 수 없고, 토큰 수가 곧 원화 비용인 것도 아니다.


---
## 필수 2. 처리 성공과 분석 정확성 구분

### 문제 2-1. 분류·질의응답·요약의 가상 응답 검증

#### 문제 설명

다음은 모두 **HTTP 200으로 응답을 받았고 status가 completed인 가상 사례**입니다. 실제 API 처리 기록이 아닙니다.

| 사례 | 상담 ID / 작업 | 가상 응답 텍스트 |
|---|---|---|
| V1 | 40005 / 상담 요건 분류 | 단일 요건 민원 |
| V2 | 40005 / 상담 요건 분류 | 이 상담은 다수 요건 민원입니다. |
| V3 | 40003 / 결합 할인 지정 대상 질의응답 | 고객 본인 |
| V4 | 40003 / 요약 | 고객은 5G 슬림+로 변경했고 기존 결합 할인은 해지되었다. VIP 등급은 그대로 유지되었다. |
| V5 | 40003 / 결합 할인 지정 대상 질의응답 | 출력 없음 |

V1·V2의 범주와 기준은 필수 1과 같습니다. V3·V5는 `고객의 결합 할인은 누구 앞으로 되어 있나요?`에 **대상만 짧게 답하도록** 요청한 상황입니다. V4는 **2문장·300자 이내**로 요금제 변경·기존 결합 할인·멤버십 등급을 요약하도록 요청한 상황입니다.

#### 요구사항

1. 질의응답 파일에서 40003의 위 질문 행을 `qa_row`, 요약 파일에서 40003의 `길이 제한 요약` 행을 `summary_row`로 조회하세요. 원문과 저장된 출력도 확인합니다.
2. 각 작업의 지시와 원문을 구분한 `qa_request`, `summary_request`를 작성하세요. 모델 값은 필수 1과 같은 설명용 표시를 사용합니다.
3. 제공된 가상 응답을 `make_demo_response()`로 만들고 생성 텍스트를 읽으세요. V5에는 `text=None`을 사용합니다. 사용량은 검토의 핵심이 아니므로 입력은 모두 1000, 출력은 차례로 8·16·4·60·0의 **가상 값**을 사용합니다.
4. `사례`, `source_id`, `record_id`, `response_id`, `status`, `가상출력`, `결과 존재`, `형식 판단`, `내용 판단`, `활용 판단` 컬럼의 `validation_table`을 작성하세요. 존재 여부는 코드로 확인하고, 형식·내용은 기준과 원문을 읽고 작성합니다.
5. 잘못된 값을 원문과 비교해 어떻게 고칠지 설명하세요. 검토용 표에는 원래 가상 출력을 유지하고, 틀린 값을 정답으로 몰래 바꿔 기록하지 않습니다.

#### 해석 질문

**Q1.** 다섯 사례 모두 completed인데 왜 그대로 활용할 수 없나요? 요청 처리·결과 존재·출력 형식·분석 정확성을 구분하여 설명하세요.

**Q2.** V1·V2는 각각 어느 부분이 문제인가요? 허용 범주 확인과 원문 검증의 차이를 설명하세요.

**Q3.** V3·V4의 원문에 맞는 답변은 무엇인가요? V5처럼 결과가 없으면 CSV의 기존 출력을 새 모델 응답인 것처럼 채워도 되나요?

**Q4.** 원문, 기존 출력, 새 응답, 응답 ID와 작업 식별자를 따로 유지하면 어떤 검증에 도움이 되나요? 검토 표와 검증을 통과한 최종 분석 결과는 어떻게 구분해야 하나요?

#### 제출 결과

- 두 작업의 요청 구조와 원문 조회
- 가상 응답을 읽어 만든 `validation_table`
- 각 오류의 수정 방향과 Q1~Q4 답변


### 풀이 안내 — 같은 완료 상태에서도 다른 문제가 생깁니다

질의응답은 같은 상담 안에도 질문이 여러 개 있을 수 있으므로 정확한 질문 문장까지 지정합니다. 요약은 ‘어떤 사실을 반드시 포함해야 하는지’와 ‘어떤 길이·문장 수로 써야 하는지’를 함께 확인하세요.

| 검토 항목 | 읽을 내용 |
|---|---|
| 응답 구조 | status와 생성 텍스트를 각각 확인 |
| 결과 존재 | 비어 있지 않은 문자열인지 확인 |
| 분류 형식 | 범주 이름 하나만 출력했는지 확인 |
| 질의응답 내용 | 원문에서 질문의 대상이 누구인지 확인 |
| 요약 내용 | 원문의 변경 사항과 유지 사항을 구분 |
| 활용 판단 | 그대로 쓸 수 있는지, 수정·검토 또는 보류가 필요한지 작성 |

제공된 사례를 반복문으로 읽는 것은 여러 번의 API 요청이 아닙니다. 형식·내용·활용 판단은 원문을 읽은 뒤 사람이 작성합니다. 오류가 있는 문장을 지우기만 하지 말고, **어떤 부분이 왜 문제인지와 어떻게 수정할지**를 함께 남기세요. 출력이 없는 경우에는 정답을 알아도 새 모델 응답처럼 채우지 않습니다.


In [4]:
# 필수 2 코드를 작성하세요.

# 1. 질의응답 행과 길이 제한 요약 행 조회
qa_question = "고객의 결합 할인은 누구 앞으로 되어 있나요?"
qa_row = get_record(qa, 40003, "의문사형", qa_question)
summary_row = get_record(summarization, 40003, "길이 제한 요약")
show_record(qa_row, include_input=True)
show_record(summary_row)
print("\n두 행의 원문 동일:", qa_row["input"] == summary_row["input"])

# 2. 작업별 요청 구성 (지시와 원문을 나눔, 전송하지 않음)
qa_request = {
    "model": "사용 가능한 모델 ID",
    "instructions": ("상담 원문을 읽고 다음 질문에 답하세요. 질문: " + qa_question
                     + " 원문에서 확인되는 대상만 짧게 답하고, 원문에 없으면 '확인 불가'라고 답하세요."),
    "input": qa_row["input"],
}
summary_request = {
    "model": "사용 가능한 모델 ID",
    "instructions": ("상담 원문을 2문장, 300자 이내로 요약하세요. 요금제 변경 처리, 기존 결합 할인 유지 여부, "
                     "멤버십 등급 변경을 반드시 포함하고 원문에 없는 사실은 추가하지 마세요."),
    "input": summary_row["input"],
}
for name, req in [("qa_request", qa_request), ("summary_request", summary_request)]:
    print(f"\n[{name}] model:", req["model"])
    print("instructions:", req["instructions"])
    print("input 글자 수:", len(req["input"]))

# 3. 가상 응답 만들기 (입력 1000, 출력 8, 16, 4, 60, 0은 모두 가상 값)
cases = [
    ("V1", class_row, "resp_V1", "단일 요건 민원", 8),
    ("V2", class_row, "resp_V2", "이 상담은 다수 요건 민원입니다.", 16),
    ("V3", qa_row, "resp_V3", "고객 본인", 4),
    ("V4", summary_row, "resp_V4",
     "고객은 5G 슬림+로 변경했고 기존 결합 할인은 해지되었다. VIP 등급은 그대로 유지되었다.", 60),
    ("V5", qa_row, "resp_V5", None, 0),
]

# 4. 형식, 내용, 활용 판단은 원문과 기준을 읽고 직접 작성
human_judgement = {
    "V1": ("범주명 하나만 출력, 형식 충족",
           "부적합: 통화 중 대기 처리와 추가 결합 할인 안내 두 주제가 있으므로 다수 요건 민원",
           "활용 불가, 원문 재검토 후 수정 필요"),
    "V2": ("형식 불충족: 범주명 외에 설명 문장이 붙음",
           "선택한 범주(다수 요건 민원)는 원문에 맞음",
           "그대로 저장 불가, 범주명만 남기도록 형식 수정 후 검토"),
    "V3": ("대상만 짧게 답함, 형식 충족",
           "부적합: 원문은 '저희 남편 앞으로 되어 있나요?' '어, 네, 맞습니다.'로 남편",
           "활용 불가, 원문 근거로 수정 필요"),
    "V4": ("2문장, 300자 이내로 형식 충족",
           "부적합: 결합 할인은 유지, 멤버십은 VIP에서 일반 플러스로 변경되었는데 반대로 씀",
           "활용 불가, 두 사실 수정 필요"),
    "V5": ("판단 불가: 출력이 없음",
           "판단 불가: 검토할 답변이 없음",
           "보류, 결과 누락으로 기록하고 원인 확인 후 재요청 여부 결정"),
}

rows = []
for case, row, rid, text, out_tok in cases:
    resp = make_demo_response(rid, text, 1000, out_tok)
    out_text = read_demo_text(resp)
    fmt, content, use = human_judgement[case]
    rows.append({
        "사례": case, "source_id": row["source_id"], "record_id": row["record_id"],
        "response_id": resp["id"], "status": resp["status"],
        "가상출력": out_text, "결과 존재": bool(out_text.strip()),
        "형식 판단": fmt, "내용 판단": content, "활용 판단": use,
    })
validation_table = pd.DataFrame(rows)
print("\n[validation_table]")
print(validation_table.to_string(index=False))

# 참고 확인: V4의 문장 수와 글자 수, 원문 속 근거
v4_text = validation_table.loc[validation_table["사례"] == "V4", "가상출력"].iloc[0]
print("\nV4 문장 수:", v4_text.count("다."), "/ 글자 수:", len(v4_text))
for p in ["저희 남편 앞으로 되어 있나요?", "따로 할인을 유지되실 거예요", "일반 플러스 등급", "VIP 혜택을 못 받는 거예요?"]:
    print(f"원문에 '{p}' 존재:", p in qa_row["input"])

# 5. 수정 방향 (검토 표의 가상출력은 바꾸지 않고 따로 기록)
corrections = pd.DataFrame([
    {"사례": "V1", "원래 가상출력": "단일 요건 민원", "수정 방향": "다수 요건 민원 (서비스 문제 + 결합 할인 상담)"},
    {"사례": "V2", "원래 가상출력": "이 상담은 다수 요건 민원입니다.", "수정 방향": "다수 요건 민원 (범주명만 남김)"},
    {"사례": "V3", "원래 가상출력": "고객 본인", "수정 방향": "남편"},
    {"사례": "V4", "원래 가상출력": v4_text,
     "수정 방향": "고객은 요금제를 5G 슬림+로 변경했고 기존 결합 할인은 그대로 유지된다. 다만 멤버십 등급은 VIP에서 일반 플러스로 바뀌었다."},
    {"사례": "V5", "원래 가상출력": "(출력 없음)", "수정 방향": "기존 output으로 채우지 않고 결과 누락으로 기록, 원인 확인 후 재요청 검토"},
])
print("\n[수정 방향]")
print(corrections.to_string(index=False))



record_id: 질의응답-6  / source_id: 40003
작업 유형: 의문사형
지시 또는 질문: 고객의 결합 할인은 누구 앞으로 되어 있나요?
상담 원문:
상담사: 일상에 즐거운 변화 LGU+ <NAME>입니다.
고객: 아, 네, 안녕하세요? 저<DATE> 날 휴대폰을 바꿨는데요.
상담사: 네.
고객: 어, 제가 요금제를 그때 대리점에서는 <DATE> 요금제를 바꿀 수 있다고 했는데 제가 예전에 <PHONE_NUMBER> 통화했을 때는 <DATE> 바꿀 수 있다고 했거든요.
상담사: 아, 네.
고객: 네, 한번 확인을 좀 해주실 수 있을까요?
상담사: 네, 확인해 보겠습니다. 입력해 주신 번호가 <NAME> 고객님 본인 맞으십니까?
고객: 아, 네, 맞아요.
상담사: 감사합니다. <DATE>부터 변경 가능하신 날짜가 맞네요. 변경하셔도 되고요.
고객: 어, 네. 근데 그게 저한테 그 대리점에서 그 전에 바꾸면 위약금이 있다는 식으로 얘기했거든요, <DATE> 전에. 네.
상담사: 아, 네, <DATE>부터, 네.
고객: 가능한 거예요?
상담사: <DATE>부터 변경하셔도 위약금 안 나오는 기간 맞습니다.
고객: 아, 네.
상담사: 매장 쪽에서 이제 아무래도 기간 자체를 조금 넉넉하게 기준 잡다 보니까 하루 이틀 정도 뒤로 말씀을 드린 거 같은데요.
고객: 아, 네.
상담사: 네, <DATE>부터 변경하실 수 있는 게 맞아요.
고객: 아, 네, 그럼 혹시 5G 다이렉트 37.5라는 그 요금제도 제가 할 수 있는 건가요?
상담사: 어, 다이렉트 요금제인데요. 고객님, 근데 이게 조건이 휴대폰 기기를 구매하실 때 저희 U+샵 부서를 통해서 개통을 해주셔야 되고요. 그리고 일부 이런 결합 할인이나 약정이 저희가 들어가시면 안 되는 요금제이기 때문에, 네.
고객: 네, 아... 그러면 안 되네요.
상담사: 네, 맞습니다.
고객: 아, 그럼 혹시 5G 슬림+ 그 <CHARGE> 짜리는 가능해요?
상담사: 네, 가능합니다.
고객: 아, 그러면 그걸로 만약에 <

### 필수 2 답변 작성란

**Q1.**

- 답변: completed는 요청 처리가 끝났다는 것만 알려 주기 때문이다. 확인 단계를 나눠 보면, 요청 처리는 다섯 사례 모두 HTTP 200, completed로 통과했다. 결과 존재는 V1부터 V4까지는 True지만 V5는 출력이 없어서 False다. 출력 형식은 V2가 범주명 대신 설명 문장으로 답해서 통과하지 못했다. 분석 정확성은 V1(단일 요건), V3(고객 본인), V4(할인 해지, VIP 유지)가 원문과 다르다. 결국 다섯 사례 중 형식과 내용을 모두 만족해서 그대로 쓸 수 있는 건 하나도 없었다. 그래서 상태만 보고 저장하면 안 되고 단계별로 따로 확인해야 한다.

**Q2.**

- 답변: V1은 "단일 요건 민원"이라서 허용 범주 검사는 통과하지만 내용이 틀렸다. 원문에 통화 중 대기 서비스 처리와 추가 결합 할인 안내라는 서로 다른 주제가 두 개 있어서 다수 요건 민원이 맞다. V2는 반대로 고른 범주(다수 요건 민원)는 맞지만 "이 상담은 다수 요건 민원입니다."처럼 문장으로 답해서 범주명만 출력하라는 형식을 어겼다. 이런 값은 `in 목록` 검사에서 False가 나오고, 그대로 저장하면 집계할 때 같은 범주로 묶이지 않는다. 허용 범주 확인은 출력 문자열이 목록에 있는지만 보는 형식 검사이고, 원문 검증은 사람이 원문을 읽고 그 범주가 실제 상담 내용에 맞는지 보는 내용 검사라서 둘 다 해야 한다.

**Q3.**

- 답변: V3의 맞는 답은 "남편"이다. 고객이 "저희 남편 앞으로 되어 있나요?"라고 묻자 상담사가 "어, 네, 맞습니다"라고 했다. V4는 결합 할인이 해지된 게 아니라 유지되고("따로 할인을 유지되실 거예요"), 멤버십은 그대로가 아니라 "일반 플러스 등급"으로 내려가서 고객이 "VIP 혜택을 못 받는 거예요?"라고 묻자 "네, 맞습니다"라고 했다. 그래서 "고객은 요금제를 5G 슬림+로 변경했고 기존 결합 할인은 그대로 유지된다. 다만 멤버십 등급은 VIP에서 일반 플러스로 바뀌었다."로 고쳐야 한다. V5처럼 결과가 없을 때 CSV의 기존 출력("남편")을 새 모델 응답처럼 채우면 안 된다. 그러면 모델이 답하지 않았는데 답한 것처럼 기록이 바뀌어서, 요청이 왜 실패했는지 원인을 찾을 수 없게 되고 모델 결과를 평가할 때도 틀린 판단을 하게 된다. 결과 누락으로 남겨 두고 원인을 확인해야 한다.

**Q4.**

- 답변: 원문을 따로 두면 응답 내용을 언제든 근거와 대조할 수 있고, 기존 출력을 따로 두면 새 응답과 비교할 수 있다(단, 기존 출력도 정답이라고 가정하지는 않는다). 새 응답을 따로 두면 모델이 실제로 뭐라고 답했는지 그대로 남고, 응답 ID와 record_id 같은 작업 식별자를 같이 두면 어떤 요청의 어떤 응답인지 추적할 수 있어서 V1과 V2처럼 같은 상담 40005를 두 번 요청한 경우도 구분된다. 이번 `validation_table`은 검토용 표라서 틀린 가상 출력도 원래대로 남기고 판단을 옆에 적었다. 검증을 통과한 최종 분석 결과는 이 검토 표와 별도의 표나 열로 만들어서, 수정된 값인지 원래 응답인지, 누가 어떤 근거로 고쳤는지가 드러나게 구분해야 한다.


---
## 과제 1. 요청 실패와 토큰 사용량 확인

### 문제 3-1. 가상 처리 기록에서 실패와 확인된 사용량 구분

#### 문제 설명

다음은 요청 상태를 관리하기 위한 **프로그램 측 가상 기록**입니다. API 원본 응답 JSON 전체가 아니며, 실제로 요청하지 않았습니다. L1·L2·L4는 40003, L3·L5는 40005의 해당 작업입니다. 같은 원문을 서로 다른 요청으로 기록하므로 `실습요청ID`로 구분합니다.

| 실습요청ID | 상담 ID / 작업 | HTTP 코드 | 응답 상태 | 새 결과 | 입력 / 출력 / 전체 토큰 |
|---|---|---:|---|---|---|
| L1 | 40003 / 할인 지정 대상 질의응답 | 200 | completed | 남편 | 1100 / 2 / 1102 |
| L2 | 40003 / 요약 | 401 | 응답 없음 | 없음 | 미확인 |
| L3 | 40005 / 상담 요건 분류 | 429 | 응답 없음 | 없음 | 미확인 |
| L4 | 40003 / 요약 | 500 | 응답 없음 | 없음 | 미확인 |
| L5 | 40005 / 상담 요건 분류 | 400 | 응답 없음 | 없음 | 미확인 |

응답 없음·결과 없음·미확인은 Python의 `None`으로 기록합니다. 실패한 요청의 사용량을 임의로 0으로 채우지 않습니다. 이 가정에서 L1만 응답 ID `resp_L1`이 있습니다.

#### 요구사항

1. 위 표로 `request_log`를 만드세요. `실습요청ID`, `source_id`, `record_id`, `HTTP코드`, `response_id`, `status`, `새결과`, `input_tokens`, `output_tokens`, `total_tokens`를 저장합니다. 작업 행은 필수 문제의 `qa_row`, `summary_row`, `class_row`를 사용합니다.
2. HTTP코드가 200이 아닌 요청만 선택하여 `failed_requests`로 확인하세요. 실패한 요청의 결과를 임의의 분류나 기존 정답으로 채우지 않습니다.
3. 실패한 요청별로 `먼저 확인할 사항`, `다음 조치`를 작성하세요. 모든 요청을 즉시 반복 호출하는 방식은 사용하지 않습니다.
4. `total_tokens`가 확인된 행만 선택하여 `known_usage`를 만들고, 입력·출력·전체 토큰의 **확인된 합계**를 구하세요. 미확인 사용량을 가진 요청 수도 출력하세요.
5. L1의 답이 원문에 맞는지 확인하고, 요청 5개를 모두 분석 완료로 볼 수 있는지 설명하세요.

**보조 구문:** `표[표["HTTP코드"] != 200]`, `표[표["total_tokens"].notna()]`, `표["total_tokens"].isna().sum()`

#### 해석 질문

**Q1.** 성공한 요청과 실패한 요청의 ID는 무엇인가요? 상담 ID만으로 L2와 L4를 구분할 수 있나요? 실패 상태와 새 결과를 왜 따로 관리해야 하나요?

**Q2.** 400·401·429·500은 각각 무엇을 먼저 확인해야 하나요? 인증 오류와 일시적인 서버 오류에 같은 재시도 방식을 적용하면 왜 문제가 되나요?

**Q3.** 확인된 입력·출력·전체 토큰 합계와 사용량 미확인 요청 수는 얼마인가요? 이 값을 전체 실제 사용량이나 원화 비용으로 확정할 수 있나요? 비용에 영향을 주는 요소를 설명하세요.

**Q4.** API 키가 공개 저장소에 올라갔다면 어떻게 해야 하나요? 상담 데이터를 실제 외부 API로 보내기 전에는 무엇을 확인해야 하나요?

#### 제출 결과

- `request_log`, `failed_requests`와 요청별 조치 표
- 확인된 사용량 합계와 미확인 요청 수
- 원문 근거 및 Q1~Q4 답변


### 풀이 안내 — 요청 수·결과 수·확인된 사용량을 나누기

과제는 필수 문제와 같은 수준에서 표 선택과 결과 해석을 연습합니다. 자동 재시도 함수를 만들거나 API 요청을 추가할 필요는 없습니다. 앞에서 선택한 `qa_row`, `summary_row`, `class_row`를 이용해 기록을 연결하세요.

| 구문 | 읽는 방법 |
|---|---|
| `표["HTTP코드"] != 200` | 각 행의 코드가 200과 다른지 비교 |
| `표[조건]` | 비교 결과가 True인 행만 선택 |
| `.copy()` | 선택한 결과를 별도 표로 복사 |
| `.notna()` | 결측이 아닌 값인지 확인 |
| `.isna()` | 결측값인지 확인 |
| `.sum()` | 숫자를 더하거나, True가 몇 개인지 셈 |

미확인 사용량은 ‘측정한 결과 0’과 다릅니다. 합계를 구한 뒤에는 어떤 행만 포함했는지와 빠진 행이 몇 개인지 함께 설명합니다. 오류 조치도 ‘다시 실행’ 한 줄로 통일하지 말고, 먼저 고쳐야 할 입력·인증 문제인지 또는 제한·서비스 상태를 확인할 상황인지 구분해 적으세요.


In [5]:
# 과제 1 코드를 작성하세요.

# 1. 가상 처리 기록 (응답 없음, 결과 없음, 미확인은 None)
log_rows = [
    ("L1", qa_row, 200, "resp_L1", "completed", "남편", 1100, 2, 1102),
    ("L2", summary_row, 401, None, None, None, None, None, None),
    ("L3", class_row, 429, None, None, None, None, None, None),
    ("L4", summary_row, 500, None, None, None, None, None, None),
    ("L5", class_row, 400, None, None, None, None, None, None),
]
request_log = pd.DataFrame([{
    "실습요청ID": lid, "source_id": row["source_id"], "record_id": row["record_id"],
    "HTTP코드": code, "response_id": rid, "status": st, "새결과": res,
    "input_tokens": it, "output_tokens": ot, "total_tokens": tt,
} for lid, row, code, rid, st, res, it, ot, tt in log_rows])
for col in ["input_tokens", "output_tokens", "total_tokens"]:
    request_log[col] = request_log[col].astype("Int64")   # 미확인은 0이 아니라 <NA>로 표시
print("[request_log]")
print(request_log.to_string(index=False))

# 2. 실패한 요청만 선택 (결과는 채우지 않음)
failed_requests = request_log[request_log["HTTP코드"] != 200].copy()

# 3. 실패 요청별 확인 사항과 다음 조치 (즉시 반복 호출하지 않음)
check_map = {
    401: ("API 키가 맞는지, 만료나 폐기되지 않았는지, 환경변수에 제대로 설정됐는지",
          "키 설정을 고친 뒤에만 다시 요청, 그 전에는 반복 요청하지 않음"),
    429: ("요청 속도 제한이나 사용 한도(크레딧)를 넘었는지",
          "잠시 기다렸다가 간격을 두고 다시 요청, 한도 문제면 계정 상태부터 확인"),
    500: ("서비스 쪽 일시적 오류인지, 서비스 상태 안내가 있는지",
          "시간을 두고 다시 요청, 같은 오류가 반복되면 기록을 남기고 보류"),
    400: ("요청 형식이 잘못됐는지(모델 ID, 항목 이름, 입력 크기 등)",
          "요청 구성을 고친 뒤 다시 요청, 그대로 반복하면 같은 오류가 남"),
}
failed_requests["먼저 확인할 사항"] = failed_requests["HTTP코드"].map(lambda c: check_map[c][0])
failed_requests["다음 조치"] = failed_requests["HTTP코드"].map(lambda c: check_map[c][1])
print("\n[failed_requests]")
print(failed_requests[["실습요청ID", "source_id", "record_id", "HTTP코드", "새결과",
                       "먼저 확인할 사항", "다음 조치"]].to_string(index=False))

# 4. 사용량이 확인된 행만 합계
known_usage = request_log[request_log["total_tokens"].notna()]
print("\n[known_usage]")
print(known_usage[["실습요청ID", "input_tokens", "output_tokens", "total_tokens"]].to_string(index=False))
print("확인된 입력 토큰 합계:", known_usage["input_tokens"].sum())
print("확인된 출력 토큰 합계:", known_usage["output_tokens"].sum())
print("확인된 전체 토큰 합계:", known_usage["total_tokens"].sum())
print("사용량 미확인 요청 수:", request_log["total_tokens"].isna().sum())

# 5. L1 답이 원문에 맞는지와 전체 완료 여부
print("\nL1 결과:", request_log.loc[0, "새결과"], "/ 기존 output:", qa_row["output"])
print("원문에 '저희 남편 앞으로 되어 있나요?' 존재:", "저희 남편 앞으로 되어 있나요?" in qa_row["input"])
print("결과가 있는 요청 수:", request_log["새결과"].notna().sum(), "/ 전체 요청 수:", len(request_log))


[request_log]
실습요청ID  source_id record_id  HTTP코드 response_id    status 새결과  input_tokens  output_tokens  total_tokens
    L1      40003    질의응답-6     200     resp_L1 completed  남편          1100              2          1102
    L2      40003     요약-10     401         NaN       NaN NaN          <NA>           <NA>          <NA>
    L3      40005     분류-23     429         NaN       NaN NaN          <NA>           <NA>          <NA>
    L4      40003     요약-10     500         NaN       NaN NaN          <NA>           <NA>          <NA>
    L5      40005     분류-23     400         NaN       NaN NaN          <NA>           <NA>          <NA>

[failed_requests]
실습요청ID  source_id record_id  HTTP코드 새결과                                  먼저 확인할 사항                                    다음 조치
    L2      40003     요약-10     401 NaN API 키가 맞는지, 만료나 폐기되지 않았는지, 환경변수에 제대로 설정됐는지     키 설정을 고친 뒤에만 다시 요청, 그 전에는 반복 요청하지 않음
    L3      40005     분류-23     429 NaN                요청 속도 제한이나 사용 한도(크레딧)를 넘었는지 잠시 기다렸

### 과제 1 답변 작성란

**Q1.**

- 답변: 성공한 요청은 L1 하나(HTTP 200, completed, 결과 "남편")이고, 실패한 요청은 L2(401), L3(429), L4(500), L5(400) 네 개다. L2와 L4는 둘 다 상담 40003의 요약(record_id 요약-10)이라서 상담 ID만으로는 구분할 수 없고, 실습요청ID로 구분해야 한다. 실패 상태와 새 결과를 따로 관리해야 하는 이유는, 결과가 비어 있는 이유가 인증 문제인지 요청 제한인지 서버 오류인지에 따라 다음 조치가 달라지기 때문이다. 결과 칸만 보고 "없음"으로 묶으면 원인이 사라지고, 또 실패한 요청에 기존 정답을 채워 넣으면 성공한 것처럼 보이게 된다. 이번에 결과가 있는 요청은 5개 중 1개뿐이었다.

**Q2.**

- 답변: 400은 잘못된 요청이라서 모델 ID, 요청 항목 이름, 입력 크기처럼 요청 구성을 먼저 확인한다. 401은 인증 실패라서 API 키가 맞는지, 만료나 폐기되지 않았는지, 환경변수에 제대로 들어갔는지를 확인한다. 429는 요청이 너무 많거나 사용 한도를 넘은 경우라서 요청 속도와 계정 한도를 확인하고 간격을 두고 다시 요청한다. 500은 서비스 쪽 오류라서 일시적인 문제인지 서비스 상태를 확인하고 시간을 두고 다시 시도한다. 인증 오류는 키를 고치지 않으면 몇 번을 다시 보내도 계속 실패하기 때문에, 서버 오류처럼 기다렸다 재시도하는 방식을 쓰면 의미 없는 요청만 반복된다. 400도 마찬가지로 요청을 고치지 않으면 같은 오류가 난다.

**Q3.**

- 답변: 확인된 사용량은 L1 한 건뿐이라서 입력 1,100, 출력 2, 전체 1,102 토큰이고, 사용량을 확인하지 못한 요청은 4개다. 이 값은 확인된 범위의 합계일 뿐이라 전체 실제 사용량이라고 확정할 수 없다. 미확인 4건을 0으로 채우면 안 되는데, 실패한 요청이 실제로 사용량이 0이었는지는 이 기록만으로 알 수 없기 때문이다. 원화 비용으로도 바로 바꿀 수 없다. 비용은 사용한 모델의 요금, 입력 토큰과 출력 토큰의 단가가 다른 점, 요청 횟수(재요청 포함), 입력 원문의 길이, 출력 길이 같은 요소에 따라 달라지고, 이번 숫자는 모두 가상 값이다.

**Q4.**

- 답변: API 키가 공개 저장소에 올라갔다면 파일에서 지우는 것만으로는 부족하다. 커밋 기록에 남아 있을 수 있어서 바로 그 키를 폐기(비활성화)하고 새 키를 발급받아야 하고, 그동안 이상한 사용 기록이 없었는지 확인해야 한다. 그리고 키는 코드나 출력에 쓰지 않고 환경변수나 비공개 입력으로만 사용한다. 상담 데이터를 실제 외부 API로 보내기 전에는 회사나 기관에서 이 데이터를 외부로 보내도 되는지 권한을 확인하고, 이름이나 전화번호 같은 개인정보가 가려져 있는지(`<NAME>`, `<DATE>` 같은 가림 표시가 유지되는지) 확인하고, API 서비스가 보낸 데이터를 저장하거나 학습에 쓰는지 같은 처리 정책도 확인해야 한다. 비용이 생길 수 있다는 점도 미리 확인한다.


### 개념 다시 정리 — 확인한 사실만큼만 결론 내리기

| 관찰한 사실 | 말할 수 있는 결론 | 추가로 필요한 확인 |
|---|---|---|
| 요청 딕셔너리를 만들었다 | 요청할 정보를 구성했다 | 실제 전송 여부와 반환 상태 |
| status가 completed이다 | 해당 응답에서 완료 상태를 확인했다 | 필요한 출력의 존재·형식·내용 |
| 텍스트 존재 검사가 True이다 | 공백만 있는 결과는 아니다 | 작업 조건과 사실 관계 |
| 허용 범주명과 일치한다 | 분류 출력의 형식 조건에 맞는다 | 올바른 범주인지 원문 대조 |
| usage 일부가 확인된다 | 그 범위의 사용량을 기록할 수 있다 | 미확인 요청과 실제 비용 정책 |
| 오류 원인과 조치를 표에 적었다 | 다음 확인 방향을 정리했다 | 실제 원인 해결과 재요청 여부 판단 |

코드가 한 작업을 끝냈을 때 무엇이 확인되었고 무엇이 아직 남았는지 구분하면 결과 해석이 명확해집니다. 실습 답변에는 데이터 출처, 확인한 조건, 근거, 남아 있는 불확실성을 연결해 적습니다. 이번에는 새로운 분석 기법이나 추가 코딩 문제를 도입하지 않고 요청·응답 확인이라는 같은 흐름을 반복해서 익힙니다.


---
## 실습 마무리

1. LLM API로 해결하려던 분석 문제와 요청·응답 흐름은 무엇인가요?
2. 원본 데이터와 새 응답을 어떻게 구분하여 연결했나요?
3. 처리 성공과 분석 정확성을 어떤 단계로 나누어 확인했나요?
4. 실패 요청과 사용량을 해석할 때 무엇을 임의로 채우면 안 되나요?
5. 이번 기본 실습에서 실제 실행한 작업과 실제로 요청하지 않은 작업은 무엇인가요?


### 실습 마무리 답변 작성란

1. 상담 원문을 LLM API로 보내 상담 요건 분류, 결합 할인 대상 질의응답, 요약을 받는 상황을 다뤘다. 흐름은 CSV에서 작업 행 선택 → model, instructions, input으로 요청 구성 → 응답의 id, status, output, usage 읽기 → 상태, 결과 존재, 형식, 내용을 나눠서 확인하는 순서였다.

2. `result_table`, `validation_table`, `request_log`에서 원문, 기존_output, 새 가상출력을 각각 다른 열에 두었다. source_id와 record_id로 어떤 상담의 어떤 작업인지, response_id와 실습요청ID로 어떤 요청과 응답인지 연결했다. 기존 출력이 새 응답과 같아도 출처가 다르니까 합치지 않았다.

3. 첫째로 status가 completed인지(요청 처리), 둘째로 공백을 뺀 텍스트가 있는지(결과 존재), 셋째로 범주명 하나인지, 2문장 300자 이내인지(출력 형식), 넷째로 원문의 실제 발화와 맞는지(분석 정확성)를 순서대로 따로 확인했다. 앞 단계가 True여도 뒤 단계는 따로 봐야 했다.

4. 실패한 요청의 결과를 기존 정답이나 그럴듯한 분류로 채우면 안 되고, 미확인 사용량을 0으로 채워도 안 된다. 실패 원인(HTTP 코드)을 남기고, 사용량 합계는 확인된 행만으로 구한 뒤 빠진 행 수(4건)를 같이 적었다.

5. 실제로 실행한 건 CSV 조회, 요청 딕셔너리 구성, 제공 함수로 가상 응답 만들기와 읽기, 상태, 존재, 범주 검사, 표 작성, 확인된 토큰 합계 계산이다. 실제 API 요청은 보내지 않았고(`RUN_LIVE_API = False`), 그래서 실제 모델 응답, 실제 토큰 사용량, 실제 비용, 실제 오류는 확인하지 않았다.


---
## 선택 실행: 단일 상담을 실제 API에 연결하기 — 제공 코드

교안의 단일 요청·응답 흐름을 확인하는 선택 실행입니다. **추가 평가 문제는 아닙니다.** 기본 필수·과제는 키 없이 끝낼 수 있습니다.

1. 실제 호출 환경은 현재 OpenAI Python SDK가 지원하는 Python 버전을 사용합니다. 필요하면 `%pip install openai`를 별도 셀에서 실행합니다.
2. 자신의 계정에서 사용할 수 있는 텍스트 모델 ID를 `LIVE_MODEL_ID`에 적습니다. 모델 ID는 비밀 키가 아닙니다.
3. API 키는 `OPENAI_API_KEY` 환경변수에 설정합니다. 없으면 코드의 비공개 입력창에 입력할 수 있습니다. 코드 셀에 키 문자열을 직접 쓰지 않습니다.
4. 데이터 활용 권한·처리 정책·개인정보를 확인하고 실제 전송하려는 경우에만 `RUN_LIVE_API = True`로 바꿉니다. 상담 40005의 원문 전체가 전송되며 과금될 수 있습니다.
5. 반환 상태·텍스트·usage를 읽고, 필수 1과 같은 네 단계로 검토합니다. 실제 응답과 토큰 수는 가상 예시와 다를 수 있습니다.

자동 재시도는 꺼 둔 제공 코드입니다. 실패하면 상태 코드 또는 연결 문제를 확인하고 원인을 해결한 뒤 필요한 경우에만 다시 실행합니다. 실행을 반복하면 같은 상담을 다시 요청하게 됩니다.

이 코드의 기본값은 False이며, 제작 과정에서는 실제 API 호출을 하지 않았습니다. 요청 구문은 [공식 텍스트 생성 안내](https://developers.openai.com/api/docs/guides/text?lang=python)와 [Python SDK 문서](https://developers.openai.com/api/reference/python)에서 확인했습니다. 교안 밖의 도구 호출·일괄 처리·출력 스키마 강제는 추가하지 않았습니다.


In [6]:
RUN_LIVE_API = False
LIVE_MODEL_ID = ""  # 실제로 사용할 수 있는 모델 ID만 입력하세요. API 키를 쓰는 곳이 아닙니다.

if not RUN_LIVE_API:
    print("실제 API 호출을 실행하지 않았습니다. 기본 실습은 가상 응답으로 진행합니다.")
else:
    import os
    from getpass import getpass
    from openai import OpenAI, APIStatusError, APIConnectionError

    if not LIVE_MODEL_ID.strip():
        raise ValueError("사용 가능한 모델 ID를 LIVE_MODEL_ID에 지정하세요.")
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("API 키를 비공개로 입력하세요: ").strip()
    if not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("API 키가 비어 있습니다. 키는 출력하지 마세요.")

    live_row = get_record(classification, 40005, "상담 요건")
    live_result = None
    client = OpenAI(max_retries=0)
    try:
        response = client.responses.create(
            model=LIVE_MODEL_ID,
            instructions=(
                "상담의 중심 요청·주제가 하나면 단일 요건 민원, 서로 구분되는 요청·주제가 "
                "두 가지 이상이면 다수 요건 민원으로 분류하세요. 같은 요청의 세부 절차 질문만 "
                "이어지면 하나로 봅니다. 두 범주 중 하나의 이름만 출력하세요."
            ),
            input=live_row["input"]
        )
        live_text = response.output_text
        print("응답 ID:", response.id)
        print("응답 상태:", response.status)
        print("생성 텍스트:", live_text)
        print("실제 반환 사용량:", response.usage)
        print("텍스트 존재:", bool(live_text.strip()))
        print("허용 범주:", live_text in ["단일 요건 민원", "다수 요건 민원"])
        live_result = pd.DataFrame([{
            "source_id": live_row["source_id"], "record_id": live_row["record_id"],
            "원문": live_row["input"], "기존_output": live_row["output"],
            "response_id": response.id, "status": response.status,
            "새_API출력": live_text, "검토상태": "원문과 조건 대조 필요"
        }])
        print("live_result는 검토 전 기록입니다. 정확성을 확인한 후 활용하세요.")
    except APIStatusError as error:
        print("요청 실패. HTTP 상태 코드:", error.status_code)
        print("결과를 임의로 채우지 않았습니다. 원인을 확인하고 재요청 여부를 결정하세요.")
    except APIConnectionError:
        print("연결 또는 응답 시간 문제입니다. 연결 상태와 이전 요청 처리 여부를 확인하세요.")
    finally:
        client.close()


실제 API 호출을 실행하지 않았습니다. 기본 실습은 가상 응답으로 진행합니다.


### 선택 실행 제공 코드 읽기 — 기본 모드와 실제 호출의 경계

이 셀은 앞 문제의 답을 작성하는 데 필수로 실행할 코드가 아닙니다. `RUN_LIVE_API = False`이면 안내 문장만 출력하고 `else` 안으로 들어가지 않습니다. 이 경우 키 입력·SDK 가져오기·실제 요청을 수행하지 않습니다. 아래 설명은 **기존 제공 코드가 어떤 순서로 동작하는지** 이해하기 위한 것입니다.

| 코드 부분 | 역할 |
|---|---|
| `RUN_LIVE_API` | 실제 호출 부분을 실행할지 정하는 스위치 |
| `LIVE_MODEL_ID` | 실제 이용 가능한 모델 이름을 담는 칸 |
| `if not LIVE_MODEL_ID.strip()` | 모델 이름이 비어 있으면 중단 |
| `os.environ.get("OPENAI_API_KEY")` | 실행 환경에 설정된 키 값을 읽음 |
| `getpass(...)` | 키가 없을 때 일반 출력에 노출하지 않고 입력받음 |
| `live_result = None` | 아직 결과 표가 없다는 초기 상태 |
| `OpenAI(max_retries=0)` | SDK 클라이언트를 만들고 자동 재시도를 끔 |
| `client.responses.create(...)` | 모델·지시·원문을 전달하는 실제 요청 위치 |
| `response.output_text` | SDK 응답에서 생성 텍스트를 읽음 |
| `response.id`, `response.status`, `response.usage` | 응답 식별자·상태·실제 반환 사용량을 각각 확인 |
| `live_result = pd.DataFrame(...)` | 원문·기존 출력·새 응답을 구분한 검토용 표 생성 |

`getpass()`로 입력한 값을 환경변수에 넣는 코드는 현재 실행 환경에서 키를 이용할 수 있게 하는 부분입니다. 키를 노트북 코드나 출력에 적어 공유하는 방식으로 바꾸지 않습니다. 모델 ID 칸에도 키를 입력하지 않습니다.

`try`는 요청과 응답 처리를 시도하는 부분입니다. `except APIStatusError`는 HTTP 오류 응답의 상태 코드를 확인하고, `except APIConnectionError`는 연결·응답 시간 문제를 안내합니다. `finally`는 해당 try 처리 후 클라이언트를 닫는 부분입니다. 이것은 실패를 성공으로 바꾸는 코드가 아닙니다.

현재 제공 코드에서 모델 이름·키 검사와 클라이언트 생성은 `try` 앞에 있습니다. 따라서 모델 이름이 비어 발생한 `ValueError` 같은 준비 단계 오류까지 뒤의 두 `except`가 모두 처리하는 구조는 아닙니다. 출력된 오류가 어느 단계에서 발생했는지 먼저 읽어야 합니다.

자동 재시도를 꺼도 사람이 셀을 다시 실행하면 새로운 요청을 보낼 수 있습니다. 연결 문제로 응답을 받지 못했다면 이전 요청의 처리 여부도 확인합니다. 실패한 경우 `live_result`를 기존 정답으로 채우지 않습니다.

실제 응답을 받아도 `검토상태`는 ‘원문과 조건 대조 필요’로 기록됩니다. 이 표를 만들었다는 것만으로 답의 정확성이 검증된 것은 아닙니다. 상태·텍스트 존재·허용 범주·원문 근거를 모두 확인합니다. 기본 예시의 분류명·응답 ID·토큰 수와 실제 반환값이 반드시 같아야 하는 것도 아닙니다.

검증 범위: 이 보강본은 실제 호출을 끈 상태에서 실행을 확인했습니다. 선택 실행의 실제 응답·사용량·비용은 측정하지 않았습니다. 기존 SDK 구문의 의미는 [OpenAI Python SDK 공식 안내](https://github.com/openai/openai-python)의 요청·응답, 오류 처리, 재시도 설정, 연결 종료 설명과 대조했습니다.
